# Assignment 11: Recommendation System
##### Author: Md Ashhar Farooqui
##### Date: 19-07-2025

## 1. Data Preprocessing
Steps:

* Load the dataset using pandas.
* Check for missing values and handle them.
* Explore the dataset (columns, datatypes, unique values).

In [ ]:
# Importing pandas
import pandas as pd

# Load the dataset
df = pd.read_csv('anime.csv')

In [ ]:
# View the first few rows
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [ ]:
# Check for missing values
print(df.isnull().sum())

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64


In [ ]:
# Fill or drop missing values as appropriate
df = df.dropna(subset=['genre', 'rating'])  # Example: drop rows with missing genre or rating

In [ ]:
# Explore data
print(df.info())
print(df['genre'].value_counts())

<class 'pandas.core.frame.DataFrame'>
Index: 12017 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12017 non-null  int64  
 1   name      12017 non-null  object 
 2   genre     12017 non-null  object 
 3   type      12017 non-null  object 
 4   episodes  12017 non-null  object 
 5   rating    12017 non-null  float64
 6   members   12017 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 751.1+ KB
None
genre
Hentai                                 816
Comedy                                 521
Music                                  297
Kids                                   197
Comedy, Slice of Life                  174
                                      ... 
Action, Hentai, Mecha, Sci-Fi, Yaoi      1
Adventure, Fantasy, Hentai               1
Hentai, Horror, Yaoi                     1
Hentai, Space                            1
Drama, Hentai, Mystery, Romance          

## 2. Feature Extraction
Steps:

* Decide on features: genres (categorical), average rating (numerical), number of episodes (numerical).
* Convert genres to a one-hot encoded format.
* Normalize numerical features if needed.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler

# Split genres into lists
df['genre_list'] = df['genre'].apply(lambda x: [g.strip() for g in x.split(',')])

In [ ]:
# One-hot encode genres
mlb = MultiLabelBinarizer()
genre_ohe = mlb.fit_transform(df['genre_list'])
genre_df = pd.DataFrame(genre_ohe, columns=mlb.classes_, index=df.index)

In [ ]:
# Convert 'episodes' to numeric, fill non-numeric with median
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

In [ ]:
# Normalize 'rating', 'episodes', 'members'
scaler = MinMaxScaler()
num_features = scaler.fit_transform(df[['rating', 'episodes', 'members']])
num_df = pd.DataFrame(num_features, columns=['rating_norm', 'episodes_norm', 'members_norm'], index=df.index)

In [ ]:

# Combine all features
features = pd.concat([genre_df, num_df], axis=1)

## Recommendation System:

* Design a function to recommend anime based on cosine similarity.
* Given a target anime, recommend a list of similar anime based on cosine similarity scores.
* Experiment with different threshold values for similarity scores to adjust the recommendation list size.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix
cos_sim = cosine_similarity(features)

In [ ]:
# Function to recommend similar anime
def recommend_anime(anime_name, top_n=5, threshold=0.5):
    if anime_name not in df['name'].values:
        print("Anime not found.")
        return []
    idx = df[df['name'] == anime_name].index[0]
    sim_scores = list(enumerate(cos_sim[idx]))
    # Filter by threshold and sort
    sim_scores = [s for s in sim_scores if s[0] != idx and s[1] >= threshold]
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_indices = [i for i, score in sim_scores[:top_n]]
    return df.iloc[top_indices][['name', 'genre', 'rating']]

In [ ]:
# Example usage
print("Recommendations for 'Steins;Gate':")
print(recommend_anime('Steins;Gate', top_n=5, threshold=0.5))

Recommendations for 'Steins;Gate':
                                                   name  \
59           Steins;Gate Movie: Fuka Ryouiki no Déjà vu   
126               Steins;Gate: Oukoubakko no Poriomania   
196   Steins;Gate: Kyoukaimenjou no Missing Link - D...   
5126                                      Under the Dog   
5525                                       Loups=Garous   

                          genre  rating  
59             Sci-Fi, Thriller    8.61  
126            Sci-Fi, Thriller    8.46  
196            Sci-Fi, Thriller    8.34  
5126   Action, Sci-Fi, Thriller    6.55  
5525  Mystery, Sci-Fi, Thriller    6.43  


## Evaluation:

* Split the dataset into training and testing sets.
* Evaluate the recommendation system using appropriate metrics such as precision, recall, and F1-score.
* Analyze the performance of the recommendation system and identify areas of improvement.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

# For evaluation, let's simulate a user-based scenario:
# We'll split the data and see if the system can recommend anime from the test set

# Split data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
# Recompute features and similarity for train set
train_features = features.loc[train_df.index]
train_cos_sim = cosine_similarity(train_features)

In [ ]:
# Build a mapping from anime name to index in train set
train_name_to_idx = {name: idx for idx, name in enumerate(train_df['name'])}

def recommend_from_train(anime_name, top_n=5, threshold=0.5):
    if anime_name not in train_name_to_idx:
        return []
    idx = train_name_to_idx[anime_name]
    sim_scores = list(enumerate(train_cos_sim[idx]))
    sim_scores = [s for s in sim_scores if s[0] != idx and s[1] >= threshold]
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_indices = [i for i, score in sim_scores[:top_n]]
    return train_df.iloc[top_indices]['name'].tolist()

In [20]:
# Evaluate: For each anime in test set, calculate the average similarity of its recommendations to other test set anime
average_similarities = []

# Build a mapping from anime name to index in the original dataframe
name_to_idx = {name: idx for idx, name in enumerate(df['name'])}

def recommend_from_df(anime_name, top_n=5, threshold=0.5):
    if anime_name not in name_to_idx:
        return []
    idx = name_to_idx[anime_name]
    sim_scores = list(enumerate(cos_sim[idx]))
    sim_scores = [s for s in sim_scores if s[0] != idx and s[1] >= threshold]
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_indices = [i for i, score in sim_scores[:top_n]]
    return df.iloc[top_indices]['name'].tolist()


for i, anime in enumerate(test_df['name']):
    recs = recommend_from_df(anime, top_n=5, threshold=0.5)
    if recs:
        # Calculate the average similarity of the recommendations to the target anime within the test set
        idx = name_to_idx[anime]
        # Filter recommendations to only include those present in the original dataframe and test set
        rec_indices = [name_to_idx[r] for r in recs if r in name_to_idx and r in test_df['name'].values]
        if rec_indices:
            similarities = [cos_sim[idx, rec_idx] for rec_idx in rec_indices]
            average_similarities.append(sum(similarities) / len(similarities))

if average_similarities:
    print(f"Average Recommendation Similarity in Test Set: {sum(average_similarities) / len(average_similarities):.2f}")
else:
    print("No recommendations generated for evaluation.")

Average Recommendation Similarity in Test Set: 0.96
